# Data Cleaning and Pre-processing
**Yanyi (Elaine) Chen**

We prepare the CDC 2022 BRFSS data to study how demographic, socioeconomic,
household, and lifestyle characteristics are associated with sleep duration.

**Setup:** For local use, keep this notebook and LLCP2022XPT.zip together.


## 1. Load the data and select columns

The publicly available [CDC 2022 BRFSS dataset](https://www.cdc.gov/brfss/annual_data/annual_2022.html)
contains **445,132 respondents**. Our raw XPT has **328 columns**.
We retain 14 columns related to the topic or needed for survey analysis:

| Original column | Clean column | Meaning |
|---|---|---|
| SLEPTIM1 | sleep_hours | Average sleep hours per 24-hour period |
| _AGEG5YR | age_group | Age interval |
| SEXVAR | sex | Respondent sex |
| INCOME3 | income_group | Annual household income interval |
| EDUCA | education_level | Highest education completed |
| EMPLOY1 | employment_status | Current employment status |
| MARITAL | marital_status | Marital/partner status |
| CHILDREN | children_group | Household children under 18 |
| EXERANY2 | physical_activity | Any leisure-time physical activity in the past month |
| _SMOKER3 | smoking_status | Current or past smoking status |
| _STATE | state | State or District of Columbia |
| _LLCPWT | survey_weight | Original final survey weight |
| _STSTR | stratum_id | Original sampling-stratum ID |
| _PSU | psu_id | Original primary-sampling-unit ID |

The ten personal/geographic characteristics describe groups whose sleep may
differ. SLEPTIM1 is used to create enough_sleep. The last three fields already
exist in the CDC data and are retained for weighted survey analysis.


In [16]:
from pathlib import Path
import shutil
import zipfile
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

PROJECT_DIR = Path.cwd()
try:
    from google.colab import drive
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    PROJECT_DIR = Path(
        "/content/drive/Shareddrives/CIS5450 Final Project/Data_Cleaning"
    )

INPUT_FILE = (Path("/content/LLCP2022.XPT") if Path("/content").is_dir()
              else PROJECT_DIR / "LLCP2022.XPT")
ZIP_FILE = PROJECT_DIR / "LLCP2022XPT.zip"
OUTPUT_DIR = PROJECT_DIR

if not INPUT_FILE.exists():
    with zipfile.ZipFile(ZIP_FILE) as archive:
        name = next(n for n in archive.namelist()
                    if n.strip().lower().endswith(".xpt"))
        with archive.open(name) as source, INPUT_FILE.open("wb") as dest:
            shutil.copyfileobj(source, dest)

columns = [
    "SLEPTIM1", "_AGEG5YR", "SEXVAR", "INCOME3", "EDUCA",
    "EMPLOY1", "MARITAL", "CHILDREN", "EXERANY2", "_SMOKER3",
    "_STATE", "_LLCPWT", "_STSTR", "_PSU",
]

reader = pd.read_sas(INPUT_FILE, format="xport", encoding="utf-8", chunksize=20000)
chunks = []
try:
    for chunk in reader:
        raw_column_count = chunk.shape[1]
        chunks.append(chunk[columns].copy())
finally:
    reader.close()

df = pd.concat(chunks, ignore_index=True).apply(pd.to_numeric, errors="coerce")
del chunks
raw_rows = len(df)
row_counts = [("Raw data", raw_rows)]

print(f"Raw data: {raw_rows:,} rows, {raw_column_count} columns")
print(f"Selected data: {len(df):,} rows, {df.shape[1]} columns")


Raw data: 445,132 rows, 328 columns
Selected data: 445,132 rows, 14 columns


**Result:** We selected **14 of the 328 original columns** and retained all
**445,132 rows** at this step.


## 2. Filter the study population and inspect unusual values

Standardize names of the **50 US states and DC**, excluding territories. Valid sleep
responses are whole hours from **1 to 24**; 77, 99, missing, and other invalid
sleep responses are excluded. Before deciding how to handle unusual values,
we inspect valid sleep durations.


In [ ]:
state_abbreviations = {
    1: "AL",   # Alabama
    2: "AK",   # Alaska
    4: "AZ",   # Arizona
    5: "AR",   # Arkansas
    6: "CA",   # California
    8: "CO",   # Colorado
    9: "CT",   # Connecticut
    10: "DE",  # Delaware
    11: "DC",  # District of Columbia
    12: "FL",  # Florida
    13: "GA",  # Georgia
    15: "HI",  # Hawaii
    16: "ID",  # Idaho
    17: "IL",  # Illinois
    18: "IN",  # Indiana
    19: "IA",  # Iowa
    20: "KS",  # Kansas
    21: "KY",  # Kentucky
    22: "LA",  # Louisiana
    23: "ME",  # Maine
    24: "MD",  # Maryland
    25: "MA",  # Massachusetts
    26: "MI",  # Michigan
    27: "MN",  # Minnesota
    28: "MS",  # Mississippi
    29: "MO",  # Missouri
    30: "MT",  # Montana
    31: "NE",  # Nebraska
    32: "NV",  # Nevada
    33: "NH",  # New Hampshire
    34: "NJ",  # New Jersey
    35: "NM",  # New Mexico
    36: "NY",  # New York
    37: "NC",  # North Carolina
    38: "ND",  # North Dakota
    39: "OH",  # Ohio
    40: "OK",  # Oklahoma
    41: "OR",  # Oregon
    42: "PA",  # Pennsylvania
    44: "RI",  # Rhode Island
    45: "SC",  # South Carolina
    46: "SD",  # South Dakota
    47: "TN",  # Tennessee
    48: "TX",  # Texas
    49: "UT",  # Utah
    50: "VT",  # Vermont
    51: "VA",  # Virginia
    53: "WA",  # Washington
    54: "WV",  # West Virginia
    55: "WI",  # Wisconsin
    56: "WY",  # Wyoming
}

df = df.loc[df["_STATE"].isin(state_abbreviations)].copy()
row_counts.append(("Keep 50 states and DC", len(df)))

df = df.loc[df["SLEPTIM1"].isin(range(1, 25))].copy()
row_counts.append(("Keep valid sleep responses", len(df)))

valid_design = (
    df["_LLCPWT"].gt(0) & df["_LLCPWT"].lt(float("inf"))
    & df["_STSTR"].between(1, 2**53 - 1) & df["_PSU"].between(1, 2**53 - 1)
    & df["_STSTR"].mod(1).eq(0) & df["_PSU"].mod(1).eq(0)
)
df = df.loc[valid_design].copy()
row_counts.append(("Keep valid weights and design IDs", len(df)))

row_summary = pd.DataFrame(row_counts, columns=["step", "rows_remaining"])
row_summary["rows_removed"] = (
    -row_summary["rows_remaining"].diff()
).fillna(0).astype(int)
display(row_summary)

display(df["SLEPTIM1"].value_counts().sort_index().rename_axis(
    "sleep_hours"
).reset_index(name="respondents"))



,step,rows_remaining,rows_removed
0,Raw data,445132,0
1,Keep 50 states and DC,435826,9306
2,Keep valid sleep responses,430578,5248
3,Keep valid weights and design IDs,430578,0


,sleep_hours,respondents
0,1.0,1126
1,2.0,1510
2,3.0,3165
3,4.0,12048
4,5.0,29196
5,6.0,93620
6,7.0,130713
7,8.0,122881
8,9.0,20932
9,10.0,10273


**Result:** Excluding territories removes **9,306 rows**. Invalid sleep
responses account for another **5,248 rows**. No additional rows have invalid
weights or design IDs, leaving **430,578 respondents**.

**Outlier decision:** Valid durations range from 1 to 24 hours; 1,519
respondents report more than 12 hours. These are unusual but valid CDC responses,
so we retain them instead of trimming by an arbitrary IQR rule. 


## 3. Clean categories, standardize names, and create enough_sleep

We keep valid CDC codes and convert invalid or missing categorical answers
to **Unknown**. CHILDREN=88 means no children and becomes 0; positive counts
are grouped into 1, 2, and 3+. State codes become postal abbreviations, and
smoking codes become English labels. 

**Category meanings after cleaning:**

| Column | Value meanings |
|---|---|
| age_group | 1=18-24; 2=25-29; 3=30-34; 4=35-39; 5=40-44; 6=45-49; 7=50-54; 8=55-59; 9=60-64; 10=65-69; 11=70-74; 12=75-79; 13=80+ |
| sex | 1=Male; 2=Female |
| income_group | Annual household income in USD: 1=below 10,000; 2=10,000 to below 15,000; 3=15,000 to below 20,000; 4=20,000 to below 25,000; 5=25,000 to below 35,000; 6=35,000 to below 50,000; 7=50,000 to below 75,000; 8=75,000 to below 100,000; 9=100,000 to below 150,000; 10=150,000 to below 200,000; 11=200,000+ |
| education_level | 1=No school/kindergarten only; 2=Grades 1-8; 3=Grades 9-11; 4=Grade 12/GED; 5=College 1-3 years/some college or technical school; 6=College 4+ years/college graduate |
| employment_status | 1=Employed for wages; 2=Self-employed; 3=Out of work for 1+ years; 4=Out of work for less than 1 year; 5=Homemaker; 6=Student; 7=Retired; 8=Unable to work |
| marital_status | 1=Married; 2=Divorced; 3=Widowed; 4=Separated; 5=Never married; 6=Member of an unmarried couple |
| children_group | Household children under 18: 0=None; 1=One; 2=Two; 3+=Three or more |
| physical_activity | Any leisure-time activity in the past month, excluding regular job duties: 1=Yes; 2=No |
| smoking_status | daily=Currently every day; sometimes=Currently some days; former=Previously smoked, not currently; never=Fewer than 100 lifetime cigarettes |
| state | Postal abbreviation, such as CA=California or PA=Pennsylvania; full mapping is in the previous code cell |
| enough_sleep | 1=7-24 hours (not short sleep); 0=1-6 hours (short sleep) |


### Outcome definition and justification

We define **enough_sleep = 1 for reported sleep of at least 7 hours** and
**0 for less than 7 hours**, after restricting responses to valid 1-24 hours.
This uses the CDC BRFSS short-sleep threshold. We export the enough_sleep.

The [CDC Chronic Disease Indicators definition](https://www.cdc.gov/cdi/indicator-definitions/sleep.html)
identifies adult short sleep as less than 7 hours per 24-hour period and uses
respondents reporting 1-24 hours as the denominator. The
[AASM/Sleep Research Society consensus (Watson et al., 2015)](https://doi.org/10.5665/sleep.4716)
also supports a 7-hour minimum for adults.

This outcome measures whether the **minimum duration threshold** is met. 


In [18]:
valid_codes = {
    "_AGEG5YR": range(1, 14), "SEXVAR": [1, 2], "INCOME3": range(1, 12),
    "EDUCA": range(1, 7), "EMPLOY1": range(1, 9), "MARITAL": range(1, 7),
    "EXERANY2": [1, 2], "_SMOKER3": range(1, 5),
}
for column, codes in valid_codes.items():
    df[column] = (
        df[column].where(df[column].isin(codes))
        .astype("Int64").astype("string").fillna("Unknown")
    )

df["_SMOKER3"] = df["_SMOKER3"].map({
    "1": "daily", "2": "sometimes", "3": "former", "4": "never",
}).fillna("Unknown")

children = df["CHILDREN"].where(df["CHILDREN"].isin(range(1, 89))).replace(88, 0)
df["CHILDREN"] = (
    children.clip(upper=3).astype("Int64").astype("string")
    .replace("3", "3+").fillna("Unknown")
)

for column in ["SLEPTIM1", "_STSTR", "_PSU"]:
    df[column] = df[column].astype("int64")
df["_STATE"] = df["_STATE"].map(state_abbreviations)

df["enough_sleep"] = df["SLEPTIM1"].ge(7).astype("int8")

df = df.rename(columns={
    "SLEPTIM1": "sleep_hours", "_AGEG5YR": "age_group", "SEXVAR": "sex",
    "INCOME3": "income_group", "EDUCA": "education_level",
    "EMPLOY1": "employment_status", "MARITAL": "marital_status",
    "CHILDREN": "children_group", "EXERANY2": "physical_activity",
    "_SMOKER3": "smoking_status", "_STATE": "state",
    "_LLCPWT": "survey_weight", "_STSTR": "stratum_id", "_PSU": "psu_id",
}).reset_index(drop=True)

category_columns = [
    "age_group", "sex", "income_group", "education_level", "employment_status",
    "marital_status", "children_group", "physical_activity", "smoking_status",
    "state",
]
df[category_columns] = df[category_columns].astype("string")
print(f"After category cleaning: {df.shape[0]:,} rows, {df.shape[1]} columns")


After category cleaning: 430,578 rows, 15 columns


**Result:** Standardizing categories removes no rows. The data have
**430,578 rows and 15 columns**: 14 retained original fields and the new
enough_sleep column.


## 4. EDA supporting cleaning: missing information and Unknown

Unknown means an invalid, refused, unknown, or missing categorical response;
it does not mean zero or the absence of a behavior. We first inspect the
Unknown percentages. Because education and physical activity have low missing
rates and removing these records causes a small sample loss, we remove rows
where either education_level or physical_activity is Unknown. Unknown values
in the other categorical inputs remain explicit categories.


In [19]:
import numpy as np

unknown_summary = pd.DataFrame({
    "column": category_columns,
    "unknown_count": df[category_columns].eq("Unknown").sum().to_numpy(),
    "unknown_percent":
        df[category_columns].eq("Unknown").mean().mul(100).round(2).to_numpy(),
})
display(unknown_summary.loc[unknown_summary["unknown_count"].gt(0)])

original_group_id = df["state"] + "_" + df["psu_id"].astype(str)
split_group_order = np.random.default_rng(42).permutation(original_group_id.unique())

rows_before = len(df)
low_missing_columns = ["education_level", "physical_activity"]
unknown_flags = df[low_missing_columns].eq("Unknown")
remove_rows = unknown_flags.any(axis=1)
display(pd.DataFrame({
    "check": ["Unknown education", "Unknown physical activity", "Unknown in both", "Unique rows removed"],
    "respondents": [
        unknown_flags["education_level"].sum(),
        unknown_flags["physical_activity"].sum(),
        unknown_flags.all(axis=1).sum(), remove_rows.sum(),
    ],
}))
df = df.loc[~remove_rows].copy().reset_index(drop=True)
row_counts.append(("Remove Unknown education or physical activity", len(df)))
row_summary = pd.DataFrame(row_counts, columns=["step", "rows_remaining"])
row_summary["rows_removed"] = (-row_summary["rows_remaining"].diff()).fillna(0).astype(int)
display(row_summary)
print(f"Removed {rows_before - len(df):,} rows ({(rows_before - len(df)) / rows_before:.2%}).")
print(f"Remaining: {len(df):,} rows and {df.shape[1]} columns.")
assert not df[low_missing_columns].eq("Unknown").any().any()
print("Remaining empty cells:", int(df.isna().sum().sum()))


,column,unknown_count,unknown_percent
0,age_group,8546,1.98
2,income_group,91798,21.32
3,education_level,2114,0.49
4,employment_status,10686,2.48
5,marital_status,4473,1.04
6,children_group,13934,3.24
7,physical_activity,937,0.22
8,smoking_status,34303,7.97


,check,respondents
0,Unknown education,2114
1,Unknown physical activity,937
2,Unknown in both,46
3,Unique rows removed,3005


,step,rows_remaining,rows_removed
0,Raw data,445132,0
1,Keep 50 states and DC,435826,9306
2,Keep valid sleep responses,430578,5248
3,Keep valid weights and design IDs,430578,0
4,Remove Unknown education or physical activity,427573,3005


Removed 3,005 rows (0.70%).
Remaining: 427,573 rows and 15 columns.
Remaining empty cells: 0


**Decision and result:** Before this removal, education has **2,114 Unknown
responses (0.49%)** and physical activity has **937 (0.22%)**. The **46**
respondents missing both are counted once, so we remove **3,005 unique rows
(0.70%)**, leaving **427,573 rows**. The sample remains well above 50,000.

Income (21.32%) and smoking (7.97%) have much more missing information before
this step. We retain Unknown for these and the other categorical inputs to
avoid broader sample loss; no mean/mode values are guessed. The small sample
loss motivates this limited removal. Remaining Unknown percentages are
reported again.


## 5. Check and export the cleaned data

We verify the 50,000-row minimum, confirm the sleep indicator matches its
definition, and export the cleaned table. The dataset remains unencoded for
the group's later analysis.


In [20]:
assert len(df) >= 50000, "The cleaned data must have at least 50,000 rows."
assert df.shape[1] == 15 and df.columns.is_unique
assert df["sleep_hours"].isin(range(1, 25)).all()
assert df["enough_sleep"].eq(df["sleep_hours"].ge(7).astype(int)).all()
assert not df.isna().any().any()

df_clean = df.copy()
output_file = OUTPUT_DIR / "brfss_2022_clean.csv"

print(f"Before cleaning: {raw_rows:,} rows, {raw_column_count} columns")
print(f"After cleaning: {df_clean.shape[0]:,} rows, {df_clean.shape[1]} columns")
print("Final columns:", ", ".join(df_clean.columns))
print("\nSleep indicator counts:")
print(df_clean["enough_sleep"].value_counts().sort_index())

display(df_clean.head())


Before cleaning: 445,132 rows, 328 columns
After cleaning: 427,573 rows, 15 columns
Final columns: sleep_hours, age_group, sex, income_group, education_level, employment_status, marital_status, children_group, physical_activity, smoking_status, state, survey_weight, stratum_id, psu_id, enough_sleep

Sleep indicator counts:
enough_sleep
0    139673
1    287900
Name: count, dtype: int64


,sleep_hours,age_group,sex,income_group,education_level,employment_status,marital_status,children_group,physical_activity,smoking_status,state,survey_weight,stratum_id,psu_id,enough_sleep
0,8,13,2,Unknown,6,7,1,0,2,never,AL,487.612985,11011,2022000001,1
1,6,13,2,5,4,2,3,0,2,never,AL,432.100273,11011,2022000002,0
2,5,8,2,10,6,7,1,0,1,never,AL,366.743194,11011,2022000003,0
3,7,Unknown,2,Unknown,4,7,1,0,1,sometimes,AL,1681.791487,11011,2022000004,1
4,9,5,2,5,5,5,1,0,1,never,AL,2111.206286,11011,2022000005,1


**Cleaned dataset:** The cleaned CSV contains **427,573 rows and 15 columns**.
enough_sleep=1 has **287,900** respondents; enough_sleep=0 has **139,673**.
These are unweighted sample counts. Unknown categories remain in the data.

The cleaned **brfss_2022_clean.csv** is saved in the same Drive folder. 


## 6. Split the cleaned data and check the training set

The exported `brfss_2022_clean.csv` contains 16 columns, including `split`, which identifies each row as `train`, `validation`, or `test`. The following cells create shared partitions for the group. We use the proposal's **60/20/20**
split. 


In [21]:

group_id = df_clean["state"] + "_" + df_clean["psu_id"].astype(str)
groups = split_group_order
a, b = int(0.60 * len(groups)), int(0.80 * len(groups))
train_data = df_clean.loc[group_id.isin(groups[:a])].copy()
val_data = df_clean.loc[group_id.isin(groups[a:b])].copy()
test_data = df_clean.loc[group_id.isin(groups[b:])].copy()

assert set(groups[:a]).isdisjoint(groups[a:b])
assert set(groups[:b]).isdisjoint(groups[b:])
assert len(train_data) + len(val_data) + len(test_data) == len(df_clean)

df_for_modeling = pd.concat([
    train_data.assign(split="train"),
    val_data.assign(split="validation"),
    test_data.assign(split="test"),
]).sort_index()

output_file = OUTPUT_DIR / "brfss_2022_clean.csv"
df_for_modeling.to_csv(output_file, index=False)

print(f"Saved to: {output_file}")
print(df_for_modeling["split"].value_counts())

display(pd.DataFrame({
    "partition": ["train", "validation", "test"],
    "respondents": [len(train_data), len(val_data), len(test_data)],
}))
class_summary = train_data.groupby("enough_sleep").agg(
    respondents=("enough_sleep", "size"),
    weight_total=("survey_weight", "sum"),
)
class_summary["sample_percent"] = (
    class_summary["respondents"] / len(train_data) * 100
).round(2)
class_summary["weighted_training_percent"] = (
    class_summary["weight_total"] / class_summary["weight_total"].sum() * 100
).round(2)
display(class_summary.drop(columns="weight_total"))


Saved to: /Users/bbelaine/Downloads/CIS5450_Final_Project/Data_preprocessing/brfss_2022_clean.csv
split
train         256605
validation     85485
test           85483
Name: count, dtype: int64


,partition,respondents
0,train,256605
1,validation,85485
2,test,85483


,respondents,sample_percent,weighted_training_percent
enough_sleep,,,
0,83713,32.62,36.06
1,172892,67.38,63.94


**Decision on class imbalance:** Training classes are approximately **67% vs
33%** (approximately **64% vs 36%** after applying the training survey
weights), so both classes have substantial representation. We retain the original
training sample rather than oversampling, undersampling, or generating synthetic
records. Resampling is not required just because the proportions differ.



**Check associations between categorical variables.** Pearson correlation on
arbitrary category codes would be misleading. We instead use **Cramer's V**,
which ranges from 0 (no association) to 1 (perfect association). This is an
unweighted training-sample diagnostic, includes Unknown as a category, and is
not a survey-adjusted hypothesis test.


In [22]:
def cramers_v(x, y):
    table = pd.crosstab(x, y).to_numpy(dtype=float)
    denominator = min(table.shape[0] - 1, table.shape[1] - 1)
    if denominator == 0:
        return 0.0
    expected = np.outer(table.sum(axis=1), table.sum(axis=0)) / table.sum()
    chi_square = ((table - expected) ** 2 / expected).sum()
    return np.sqrt(chi_square / (table.sum() * denominator))

associations = []
for i, first in enumerate(category_columns):
    for second in category_columns[i + 1:]:
        associations.append({
            "first": first, "second": second,
            "cramers_v": cramers_v(train_data[first], train_data[second]),
        })
association_summary = pd.DataFrame(associations).sort_values(
    "cramers_v", ascending=False
).reset_index(drop=True)
display(association_summary.head(5).round(3))


,first,second,cramers_v
0,employment_status,children_group,0.336
1,age_group,employment_status,0.313
2,age_group,marital_status,0.292
3,age_group,children_group,0.283
4,children_group,smoking_status,0.220


**Decision on redundant variables:** The largest observed association is
approximately **0.336**, between employment status and children group.
We retain all ten characteristics because this diagnostic does not indicate
near-perfect duplication. This does not prove the absence of all
multicollinearity.


## 7. Encode categorical variables for modeling

We use ordinal encoding for variables with a meaningful order and one-hot
encoding for variables without a meaningful order.

| Encoding | Variables | Reason |
|---|---|---|
| Ordinal | age_group, income_group, education_level, children_group | Categories have a natural increasing order |
| One-hot | sex, employment_status, marital_status, physical_activity, smoking_status, state | Categories do not have a single meaningful numerical order |

### Ordered variables and Unknown values

Known categories are assigned increasing integers:
- age_group: 1–13, from ages 18–24 to 80+.
- income_group: 1–11, from the lowest to highest income interval.
- education_level: 1–6, from the lowest to highest education category.
- children_group: 0, 1, 2, and 3+ children become codes 1, 2, 3, and 4.

Unknown is not an ordered level. For each ordinal variable, we use a
placeholder of 0 together with a separate `_unknown` indicator.
This distinguishes missing information from known low categories.

These codes represent ordered categories, not actual ages, dollar amounts,
or equally spaced measurements. When used numerically in logistic regression,
they impose a linear effect per rank step, which is a modeling assumption.

### Unordered variables and shared encoding

We create binary indicators for nominal categories and omit one reference
category per variable to avoid exact dummy-variable redundancy with an intercept.
Unknown remains a separate indicator where applicable.

Category definitions and ordinal mappings are fixed from the CDC codebook.
The same mappings are applied to all three existing partitions, without
learning categories or choosing reference groups from validation or test data.

Only the ten selected characteristics are encoded as model inputs.
The outcome `enough_sleep`, survey weights, survey-design IDs, and `split`
remain separate. We exclude `sleep_hours` from model inputs because it directly
determines the outcome.

We save a separate encoded CSV while preserving the original cleaned CSV.


In [23]:
import pandas as pd

# Ordered levels follow the CDC codebook, not alphabetical sorting.
ordinal_levels = {
    "age_group": [str(i) for i in range(1, 14)],
    "income_group": [str(i) for i in range(1, 12)],
    "education_level": [str(i) for i in range(1, 7)],
    "children_group": ["0", "1", "2", "3+"],
}

# These categories have no single meaningful numerical order.
# The first level in each list is the one-hot reference category.
nominal_levels = {
    "sex": ["1", "2", "Unknown"],
    "employment_status": [str(i) for i in range(1, 9)] + ["Unknown"],
    "marital_status": [str(i) for i in range(1, 7)] + ["Unknown"],
    "physical_activity": ["1", "2"],
    "smoking_status": ["never", "former", "sometimes", "daily", "Unknown"],
    "state": sorted(state_abbreviations.values()),
}

def encode_features(data):
    X = pd.DataFrame(index=data.index)

    # Known ordered categories become 1, 2, ..., K.
    # Unknown uses a placeholder of 0 plus its own binary indicator.
    for column, levels in ordinal_levels.items():
        values = data[column].astype("string")
        assert values.isin(levels + ["Unknown"]).all(), column

        mapping = {
            level: rank
            for rank, level in enumerate(levels, start=1)
        }
        X[column] = values.map(mapping).fillna(0).astype("int8")
        X[column + "_unknown"] = values.eq("Unknown").astype("int8")

    # Fixed CDC categories give every partition the same dummy columns.
    nominal = pd.DataFrame(index=data.index)

    for column, levels in nominal_levels.items():
        values = data[column].astype("string")
        assert values.isin(levels).all(), column
        nominal[column] = pd.Categorical(values, categories=levels)

    one_hot = pd.get_dummies(nominal, drop_first=True, dtype="int8")

    return pd.concat([X, one_hot], axis=1)


# These mappings are fixed in advance; nothing is fitted on validation/test.
encoded_features = encode_features(df_for_modeling)
feature_columns = encoded_features.columns.tolist()

# Preserve the existing partitions and row alignment.
X_train = encoded_features.loc[train_data.index].copy()
X_val = encoded_features.loc[val_data.index].copy()
X_test = encoded_features.loc[test_data.index].copy()

y_train = train_data["enough_sleep"].copy()
y_val = val_data["enough_sleep"].copy()
y_test = test_data["enough_sleep"].copy()

w_train = train_data["survey_weight"].copy()
w_val = val_data["survey_weight"].copy()
w_test = test_data["survey_weight"].copy()

# Keep the outcome and survey fields separate from predictive features.
metadata_columns = [
    "enough_sleep", "survey_weight", "stratum_id", "psu_id", "split",
]

df_encoded = pd.concat([
    encoded_features,
    df_for_modeling[metadata_columns],
], axis=1)

# Check completeness, consistent columns, and matching row order.
assert len(df_encoded) == len(df_for_modeling)
assert not df_encoded.isna().any().any()
assert X_train.columns.equals(X_val.columns)
assert X_train.columns.equals(X_test.columns)
assert X_train.index.equals(y_train.index)
assert X_val.index.equals(y_val.index)
assert X_test.index.equals(y_test.index)

# Prevent target leakage and accidental use of survey IDs as predictors.
assert set(feature_columns).isdisjoint(
    metadata_columns + ["sleep_hours"]
)

# Write a new file; keep the original cleaned CSV unchanged.
encoded_file = OUTPUT_DIR / "brfss_2022_encoded.csv"
df_encoded.to_csv(encoded_file, index=False)

display(pd.DataFrame({
    "column": list(nominal_levels),
    "reference_category": [
        levels[0] for levels in nominal_levels.values()
    ],
}))

print("Number of model features:", len(feature_columns))
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)
print("Encoded dataset:", df_encoded.shape)
print("Saved to:", encoded_file)

display(X_train.head())

,column,reference_category
0,sex,1
1,employment_status,1
2,marital_status,1
3,physical_activity,1
4,smoking_status,never
5,state,AK


Number of model features: 79
Training: (256605, 79)
Validation: (85485, 79)
Test: (85483, 79)
Encoded dataset: (427573, 84)
Saved to: /Users/bbelaine/Downloads/CIS5450_Final_Project/Data_preprocessing/brfss_2022_encoded.csv


,age_group,age_group_unknown,income_group,income_group_unknown,education_level,education_level_unknown,children_group,children_group_unknown,sex_2,sex_Unknown,employment_status_2,employment_status_3,employment_status_4,employment_status_5,employment_status_6,employment_status_7,employment_status_8,employment_status_Unknown,marital_status_2,marital_status_3,marital_status_4,marital_status_5,marital_status_6,marital_status_Unknown,physical_activity_2,smoking_status_former,smoking_status_sometimes,smoking_status_daily,smoking_status_Unknown,state_AL,state_AR,state_AZ,state_CA,state_CO,state_CT,state_DC,state_DE,state_FL,state_GA,state_HI,state_IA,state_ID,state_IL,state_IN,state_KS,state_KY,state_LA,state_MA,state_MD,state_ME,state_MI,state_MN,state_MO,state_MS,state_MT,state_NC,state_ND,state_NE,state_NH,state_NJ,state_NM,state_NV,state_NY,state_OH,state_OK,state_OR,state_PA,state_RI,state_SC,state_SD,state_TN,state_TX,state_UT,state_VA,state_VT,state_WA,state_WI,state_WV,state_WY
1,13,0,5,0,4,0,1,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,1,0,1,4,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,13,0,0,1,4,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,13,0,8,0,6,0,1,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,13,0,7,0,4,0,1,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


**Result:** The ten original characteristics are represented by 79 numerical
model features, including ordinal codes, Unknown indicators, and one-hot
indicators. All three partitions share the same feature columns.

Encoding removes no observations. The encoded dataset contains 427,573 rows
and 84 columns: 79 predictive features plus the outcome, survey weight,
two survey-design IDs, and split label.

The separate `brfss_2022_encoded.csv` is saved in OUTPUT_DIR. The original
cleaned CSV remains available for EDA and interpretation.

Subsequent modeling should use X_train/X_val/X_test as inputs and
y_train/y_val/y_test as outcomes. Survey weights are available separately
as w_train/w_val/w_test.